In [1]:
import numpy as np
import pyro


In [2]:
from pyro import sample

In [4]:
import torch
from pyro.distributions import Normal, Categorical

Causal dag is. X -> Y with two continuous confounders Z1 and Z2, each of which influences both X and Y.  We are given P(X| Z1, Z2) (propensity score) by the exercise as follows:

In [5]:
def pXgivenZ(x, z1, z2):
  prob_X = torch.tensor([
              [0.15, 0.85],
              [0.90, 0.10]
          ])
  if z1 + z2 <= 0:
    return prob_X[0][x]
  else:
    return prob_X[1][x]

In [6]:
pXgivenZ(1, -1, -1)

tensor(0.8500)

We are also given a model for the joint distribution of X, Y and Z1, Z2 as follows:

In [11]:
def model():
  z1 = sample('Z1', Normal(0, 1))
  z2 = sample('Z2', Normal(0, 1))
  prob_X = torch.tensor([
              [0.15, 0.85],
              [0.90, 0.10]
          ])
  prob_Y = torch.tensor([[[0.068, 0.932],
                          [0.267, 0.733]],
                         [[0.131, 0.869],
                          [0.313, 0.687]]])
  if z1 + z2 <= 0:
    x = sample('X', Categorical(probs=prob_X[0]))
    y = sample('Y', Categorical(probs=prob_Y[x][0]))
  else:
    x = sample('X', Categorical(probs=prob_X[1]))
    y = sample('Y', Categorical(probs=prob_Y[x][1]))
  return x, y, z1, z2

In [13]:
samples = []
sample_size = 1000
trace_handler = pyro.poutine.trace(model)
for i in range(sample_size):
    trace = trace_handler.get_trace()
    x = trace.nodes['X']['value']
    y = trace.nodes['Y']['value']
    z1 = trace.nodes['Z1']['value']
    z2 = trace.nodes['Z2']['value']
    p = np.exp(trace.log_prob_sum())
    samples.append({'X': x, 'Y': y, 'Z1': z1, 'Z2': z2, 'p':p})

/var/folders/63/799zv0dx3jl7q4xq7kz5f9fc0000gn/T/ipykernel_5204/1332612306.py:10: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  p = np.exp(trace.log_prob_sum())


Now we are suppose to compute the causal effect using IPW approach:

1. For E(Y|do(X=1)), filter the samples to those where X = 1.
2. For each of the remaining samples, take the values of Z1 and Z2 (and X=1) to calculate the propensity score.
3. Reweight the sample as the sample probability divided by the propensity score.
4. Use these weights to calculate the weighted mean of Y across the samples (or resample with replacement using these weights and take the regular mean). This will give you an estimate of E(Y|do(X=1))
5. Repeat steps 1-4 for E(Y|do(X=0))


In [ ]:
 
def E_do_X(x):
    # Step 1 , is done on each line.
    propensities = np.array([pXgivenZ(x, s['Z1'], s['Z2']) for s in samples if s['X'] == x]) # step 2
    sample_probs = np.array([s['p'] for s in samples if s['X'] == x]) # step 3a
    ys = np.array([s['Y'] for s in samples if s['X'] == x]) 

    weights = sample_probs / propensities # step 3b
    return np.average(ys, weights=weights) # step 4
 

Now we can estimate the effect:

In [49]:
E_do_X(1) - E_do_X(0) # step 5

np.float64(0.07020888943697867)